# 06a — Strategy 1: Inventory-Led Model Transition
### Tested with Reinforcement Learning (PPO)

**Where this sits in the case study**

Notebooks 01-05 *diagnosed* why Instamart's path to profitability lags Blinkit. Notebook 06
*tests the cures*. This is the first of three strategy simulations:

- **06a (this notebook)** — Should Swiggy transition Instamart to Blinkit's inventory-led model, and if so, how fast? → Reinforcement Learning
- **06b** — Should Swiggy cross-sell its 18.3M food-delivery users into Instamart? → Uplift modeling + Monte Carlo
- **06c** — Should Swiggy hold store expansion and grow density first? → System Dynamics + deep-uncertainty analysis
- **06d** — How do the three strategies compare and sequence?

**The question this notebook answers**

Branch 4 (logistics) found that 90% of Blinkit's NOV runs through an inventory-led model worth
a stated 50-70bps of margin, and that Swiggy's own shareholder vote to adopt the same model
*failed by 2.65pp* (72.35% vs. the 75% needed). So the lever exists and is proven by a
competitor. The open question is one of **pace and sequencing**: if Swiggy did get to transition,
how aggressively should it push the inventory-led share each quarter, given that the transition
costs capex, and capex is also what funds the density growth that drives margin?

That is a sequential decision problem under a budget constraint — exactly what reinforcement
learning is for. We frame it as a Markov Decision Process, build a calibrated simulation
environment, and train a PPO agent to learn the optimal transition policy.

**Honesty flag up front (read before trusting any number below)**

This environment is calibrated to the disclosed aggregates from Notebook 01 (1,025 orders/store/day,
~25% of stores profitable, -1.8% contribution margin, 60bps inventory-model uplift) and to the
n=1,143 *simulated* store network from Notebook 02. The learned policy is therefore optimal
**within a simplified model**, not a forecast of what will happen in reality. The transition cost,
the density→margin functional form, and the competitive-response coefficients are all assumptions,
flagged inline where they appear. The right way to read the output is *"this is the methodology for
how you'd find the optimal transition pace once you have real store-level data"* — not *"Swiggy
should push inventory-led share by exactly X pp in quarter 3."*


In [ ]:

# If running for the first time, install the RL dependencies:
# pip install "gymnasium>=0.29" "stable-baselines3>=2.3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, Optional

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.logger import configure
import os
import torch
import gc

# --- Resource caps so training never hoards your laptop ---
# Cap CPU threads (your Ryzen 5 4600H has 6 cores / 12 threads; leave headroom for other work).
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
torch.set_num_threads(4)
torch.backends.cudnn.benchmark = False

# --- Device choice: CPU is the RIGHT call here, and here's why ---
# PPO with a small MlpPolicy does tiny matrix multiplies. On a GTX 1650 the cost of moving
# each minibatch CPU<->GPU outweighs the compute saving, and the environment stepping (which
# dominates wall-clock here) runs on CPU regardless. SB3's own docs note PPO/A2C with MLP
# policies are usually FASTER on CPU. GPU only helps SB3 for CnnPolicy on image observations.
# We still detect CUDA (so you can flip USE_GPU=True to experiment), but default to CPU.
USE_GPU = False
device = "cuda" if (USE_GPU and torch.cuda.is_available()) else "cpu"
print(f"Torch sees CUDA: {torch.cuda.is_available()}"
      + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print(f"Training device: {device}  (MLP-PPO is intentionally on CPU \u2014 see comment above)")
if device == "cuda":
    torch.cuda.empty_cache()

PROCESSED = Path("../data/processed")
MODELS = Path("../models/strategy1")
MODELS.mkdir(parents=True, exist_ok=True)

RNG_SEED = 42
np.random.seed(RNG_SEED)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Brand palette, consistent with the rest of the case study / deck
SWIGGY = "#FC8019"; BLINKIT = "#0C9D61"; ZEPTO = "#8025FB"; NAVY = "#1B2A4A"


## 2.0 Calibrated Baseline

Every constant below is pulled from `master_metrics.csv` (Notebook 01) or the simulated store
network (Notebook 02). Confidence tags: **D** = company-disclosed, **E** = analyst-estimated,
**DV** = derived. Items tagged E or DV are assumptions the policy rests on, flagged inline.

> **Critical assumption — flag this in a portfolio walkthrough:** `inv_model_total_bps` (240 bps)
> is the single most load-bearing assumption in this notebook. The CFO-stated 60 bps is a
> *per-order* disclosed fact; 240 bps models the **full structural margin contribution** of the
> inventory-led model (warehouse efficiency, supplier terms, reduced spoilage) — not individually
> disclosed. Section 8 sweeps the reward weights but not this parameter; a real engagement would
> stress it too.

In [ ]:

# Load the disclosed figures straight from Notebook 01's output so this notebook
# never hard-codes a number that already lives in the master table.
master = pd.read_csv(PROCESSED / "master_metrics.csv")

def lookup(company, metric, default=None):
    hit = master[(master.company == company) & (master.metric == metric)]
    return float(hit["value"].iloc[0]) if len(hit) else default

BASELINE = {
    # ── Swiggy Instamart ───────────────────────────────────────────────────────
    "avg_density":                  lookup("Swiggy Instamart", "Orders per Store per Day", 1025),   # D
    "density_ceiling":              2000,     # D  — Swiggy stated 2,000+ capacity ceiling
    "breakeven_density":            1552,     # DV — derived in Notebook 02 (simulated regression)
    "pct_profitable_stores":        0.25,     # D  — Swiggy Q2 FY26 shareholder letter
    "contribution_margin":         -0.018,    # D  — Q4 FY26, expressed as a fraction
    "market_share":                 0.24,     # E  — Datum Intelligence / Reuters, Jan 2026
    "pct_inventory_led":            0.10,     # DV — Swiggy is marketplace-led; Blinkit 90% = ceiling
    "capex_reserve_cr":             500,      # E  — approximate quarterly capex envelope
    "total_stores":                 1143,     # D  — Eternal/Swiggy disclosures (Notebook 02 network)
    # ── Blinkit reference figures ──────────────────────────────────────────
    "blinkit_density":              lookup("Blinkit", "Orders per Store per Day", 1337),   # E — Nomura note
    "blinkit_market_share":         0.46,     # E  — Datum Intelligence / Reuters, Jan 2026
    "blinkit_contribution_margin":  0.012,    # E  — analyst estimate; used as the target end-state
    # ── Inventory model economics ─────────────────────────────────────────
    "inv_uplift_bps":               60,       # D  — CFO-stated 50–70 bps midpoint (per-order disclosed fact)
    "inv_model_total_bps":          240,      # E  — *** MODELLING ASSUMPTION ***
                                              #      Estimated full structural margin contribution of the
                                              #      inventory-led model: warehouse efficiency, supplier
                                              #      terms, reduced spoilage, pooled last-mile economics.
                                              #      Calibrated so full transition (10%→90%) bridges
                                              #      ~200 bps of the Swiggy/Blinkit margin gap.
                                              #      No individual public source — single most load-bearing
                                              #      assumption in the model.
    "capex_per_5pp_cr":             80,       # E  — ₹80 cr per 5pp transition step (estimate)
}

for k, v in BASELINE.items():
    print(f"  {k:30s} = {v}")

## 3.0 Environment Design — State, Action, Reward

**State (6 continuous variables).** Everything the decision-maker can observe each quarter:

| Variable | Range | Meaning |
|---|---|---|
| `density_norm` | 0–1 | orders/store/day ÷ ceiling |
| `pct_inventory_led` | 0–0.9 | share of NOV running through the inventory model |
| `contribution_margin` | −0.10 to +0.10 | fractional contribution margin |
| `market_share` | 0–1 | share of the quick-commerce market |
| `capex_norm` | 0–1 | capex reserve ÷ ₹1,000 cr |
| `quarter` | 0–11 | three-year horizon |

**Action (Discrete, 5 choices).** How hard to push the inventory transition this quarter:

| Action | Effect on inventory-led share |
|---|---|
| 0 — Hold | no change |
| 1 — Slow push | +5 pp |
| 2 — Moderate push | +10 pp |
| 3 — Aggressive push | +15 pp |
| 4 — Retreat | −5 pp (de-risk if capex is tight) |

Discrete (not continuous) on purpose: the learned policy reads as a sentence — *"in quarter 3 the
agent chooses Moderate push"* — which is far easier to defend in an interview than a continuous
control signal.

**Reward.** A weighted blend of the three things a quick-commerce operator actually cares about:
margin (50%), competitive position (30%), and capex sustainability (20%). The weights are a
modelling choice; Section 8 re-runs training across different weightings to show how sensitive the
policy is to them.

**Episode.** 12 quarters. Terminates early if contribution margin stays below −6% for three
consecutive quarters (a business-distress stop).

## 4.0 Custom Gym Environment

The `step()` method encodes the causal model. Read the inline comments as the assumptions ledger —
each relationship is either disclosed, calibrated to a disclosed figure, or an explicit estimate.

**Key causal relationships modelled:**
1. `pct_inventory_led` ↑ → margin ↑ (LEVEL effect proportional to current inventory share)
2. `density` above breakeven → margin ↑ (log-linear; calibrated to n=1,143 simulation)
3. Positive `margin` → capex regenerates → density growth funded (reinforcing loop)
4. Losing market share vs Blinkit → competitive drag on margin (balancing force)
5. Transition itself costs capex (₹80 cr per 5 pp moved — estimate)

In [ ]:

class InstamartTransitionEnv(gym.Env):
    """
    Simulates Swiggy Instamart's quarter-by-quarter decision on how fast to transition
    from a marketplace-led to an inventory-led fulfilment model.

    Causal relationships modelled (all calibrated to Notebook 01 / 02 figures):
      1. Inventory-led share up  -> margin up (LEVEL effect; driven by inv_model_total_bps).
      2. Density above breakeven -> margin up (log-linear; calibrated to the n=1,143 sim).
      3. Margin up               -> capex regenerates -> density growth funded.
      4. Losing share vs Blinkit -> promo spend up    -> margin drag (competitive response).
      5. Transition itself costs capex (estimated ₹80 cr per 5 pp moved).
    """
    metadata = {"render_modes": []}

    def __init__(self, config: Optional[Dict] = None):
        super().__init__()
        self.cfg = config or {}

        self.observation_space = spaces.Box(
            low =np.array([0.0, 0.0, -0.10, 0.0, 0.0,  0.0], dtype=np.float32),
            high=np.array([1.0, 0.9,  0.10, 1.0, 1.0, 11.0], dtype=np.float32),
        )
        self.action_space = spaces.Discrete(5)
        self._inv_delta = {0: 0.00, 1: 0.05, 2: 0.10, 3: 0.15, 4: -0.05}

        # Reward weights (tunable — see Section 8)
        self.w_margin = self.cfg.get("w_margin", 0.50)
        self.w_share  = self.cfg.get("w_share",  0.30)
        self.w_capex  = self.cfg.get("w_capex",  0.20)
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.q       = 0
        self.density = BASELINE["avg_density"]
        self.pct_inv = BASELINE["pct_inventory_led"]
        self.margin  = BASELINE["contribution_margin"]
        self.share   = BASELINE["market_share"]
        self.capex   = BASELINE["capex_reserve_cr"]
        self._distress = 0
        return self._obs(), {}

    def step(self, action: int):
        # 1. Apply the transition decision; pay its capex cost
        d_inv        = self._inv_delta[int(action)]
        self.capex   = max(0.0, self.capex - abs(d_inv) * BASELINE["capex_per_5pp_cr"])
        self.pct_inv = float(np.clip(self.pct_inv + d_inv, 0.0, 0.90))

        # 2. Margin update
        #    (a) Inventory-model margin — LEVEL effect.
        #        The disclosed 60 bps (inv_uplift_bps) is the per-order CFO-stated fact.
        #        inv_model_total_bps (240 bps) is the ESTIMATED full structural contribution
        #        of running an inventory-led model (warehouse efficiency, supplier terms,
        #        reduced spoilage) — the single most load-bearing assumption in this model.
        #        At pct_inv=0.10 (start), 24 bps is already embedded in baseline margin;
        #        subtracting it gives base_no_inv (hypothetical margin at zero inventory share).
        base_no_inv      = (BASELINE["contribution_margin"]
                            - BASELINE["pct_inventory_led"]
                            * (BASELINE["inv_model_total_bps"] / 10_000))
        inv_margin_level = self.pct_inv * (BASELINE["inv_model_total_bps"] / 10_000)

        #    (b) Density effect: log-linear above breakeven (1,552 orders/store/day)
        density_above_be = max(0.0, self.density - BASELINE["breakeven_density"])
        density_effect   = 0.006 * np.log1p(density_above_be / BASELINE["breakeven_density"])

        #    (c) Competitive drag: losing share → promo spend ↑ → margin ↓
        competitive_drag = -0.003 * max(0.0, BASELINE["blinkit_market_share"] - self.share)

        self.margin = float(np.clip(
            base_no_inv + inv_margin_level + density_effect + competitive_drag,
            -0.10, 0.10
        ))

        # 3. Density update: positive margin regenerates capex, which funds density growth.
        #    Calibrated so a healthy business grows ~40–70 orders/store/day per quarter,
        #    consistent with reaching the 1,552 breakeven threshold in 6–8 quarters
        #    under a good policy (vs 264 quarters under the prior delta-based formula).
        self.capex  += max(0.0, self.margin) * 200.0
        density_growth = 20.0 + max(0.0, self.capex - 200.0) / 500.0 * 80.0
        self.density = float(min(self.density + density_growth, BASELINE["density_ceiling"]))

        # 4. Market-share update: driven by the density gap vs Blinkit
        gap        = (self.density - BASELINE["blinkit_density"]) / BASELINE["blinkit_density"]
        self.share = float(np.clip(self.share + 0.005 * gap, 0.10, 0.60))

        # 5. Reward + termination
        reward = self._reward()
        self._distress = self._distress + 1 if self.margin < -0.06 else 0
        self.q += 1
        terminated = self._distress >= 3
        truncated  = self.q >= 12
        return self._obs(), reward, terminated, truncated, self._info()

    def _obs(self):
        return np.array([
            self.density / BASELINE["density_ceiling"],
            self.pct_inv, self.margin, self.share,
            self.capex / 1000.0, float(self.q),
        ], dtype=np.float32)

    def _reward(self):
        r_margin = self.w_margin * self.margin * 100.0
        r_share  = self.w_share  * (self.share - BASELINE["market_share"]) * 10.0
        r_capex  = self.w_capex  * (self.capex / 500.0 - 1.0)
        return float(r_margin + r_share + r_capex)

    def _info(self):
        return dict(density=self.density, pct_inv=self.pct_inv, margin=self.margin,
                    share=self.share, capex=self.capex, quarter=self.q)

In [ ]:

# Sanity-check the environment against the Gym API before training on it.
env = InstamartTransitionEnv()
check_env(env, warn=True)
print("Environment passes the Gymnasium API check.")
print("Observation space:", env.observation_space)
print("Action space:     ", env.action_space)


## 5.0 Reward Function Deep-Dive

The reward is a **weighted blend of three business objectives** the agent optimises each quarter.
This decomposition is explicit so the reward is interpretable — you can trace *why* the agent
prefers one action over another.

| Component | Variable | Weight | Scaling | What it captures |
|---|---|---|---|---|
| Contribution margin | `margin` | **50%** | ×100 (→ % pts) | Core profitability — the primary gating signal |
| Market share Δ vs baseline | `share − 0.24` | **30%** | ×10 | Competitive position vs Blinkit / Zepto |
| Capex reserve health | `capex / 500 − 1` | **20%** | normalised | Operational sustainability |

```python
def _reward(self) -> float:
    r_margin = self.w_margin * self.margin * 100        # scale to ~[-10, +10]
    r_share  = self.w_share  * (self.share - 0.24) * 10
    r_capex  = self.w_capex  * (self.capex / 500.0 - 1.0)
    return float(r_margin + r_share + r_capex)
```

**Why these weights?** The 50/30/20 split reflects a creditor-framing: margin is the gating
constraint (Swiggy needs positive contribution margin to unlock the path to EBITDA break-even),
share prevents the agent from ignoring competitive deterioration, and capex ensures it does not
pursue short-term margin at the cost of operational insolvency.

**Reward shaping is a modelling choice, not a ground truth.** Section 8 re-trains the agent
across three weight configurations (margin-focused 70/20/10, base case 50/30/20, share-focused
30/50/20) and compares the resulting policies — showing whether the learned strategy is robust
to this assumption or sensitive to it.

### 5.1 Pre-Training Baselines

Before training the agent, establish what a *do-nothing* policy (always Hold) and a *maximally
aggressive* policy (always +15 pp) produce. If PPO cannot beat these two extremes, the learning
added nothing of value.

In [ ]:

def run_fixed_policy(action_fn, n_episodes=200):
    env = InstamartTransitionEnv()
    finals = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        total_r = 0.0
        while not done:
            a = action_fn(obs)
            obs, r, term, trunc, info = env.step(a)
            total_r += r
            done = term or trunc
        finals.append({**info, "total_reward": total_r})
    return pd.DataFrame(finals)

always_hold       = run_fixed_policy(lambda obs: 0)
always_aggressive = run_fixed_policy(lambda obs: 3)

print("Always-Hold      :  median final margin = {:+.3f},  median reward = {:.1f}".format(
    always_hold["margin"].median(), always_hold["total_reward"].median()))
print("Always-Aggressive:  median final margin = {:+.3f},  median reward = {:.1f}".format(
    always_aggressive["margin"].median(), always_aggressive["total_reward"].median()))


## 6.0 Agent Training (PPO)

`gamma=0.97` because steps are quarters, so a slightly heavier discount than the usual 0.99
keeps the agent from over-weighting events three years out that the model cannot credibly predict.

**Observability:** `verbose=1` on PPO shows rollout/train tables each update; `EvalCallback`
prints a summary every `eval_freq` steps; the SB3 logger is wired to stdout + CSV + TensorBoard
so you can monitor live with `tensorboard --logdir ../models/strategy1/tb`.

**Budget:** 400k timesteps — enough for the policy to converge on this small env, light enough
to finish in a few minutes on CPU without locking up the laptop.

In [ ]:

# Monitor-wrap both envs so episode rewards/lengths are reported correctly
# (this removes the 'not wrapped with Monitor' warning and fixes EvalCallback stats).
train_env = Monitor(InstamartTransitionEnv())
eval_env  = Monitor(InstamartTransitionEnv())

# Eval callback with verbose=1 -> prints periodic evaluation summaries during training
eval_cb = EvalCallback(
    eval_env, best_model_save_path=str(MODELS),
    log_path=str(MODELS), eval_freq=5000,
    n_eval_episodes=20, deterministic=True, verbose=1,
)

# SB3 logger -> stdout + CSV + TensorBoard scalars
logger = configure(str(MODELS / "logs"), ["stdout", "csv", "tensorboard"])

# Memory-friendly rollout buffer: n_steps=1024 keeps the buffer (1024 x obs) tiny,
# so RAM stays low and other work on the laptop isn't disrupted.
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

model = PPO(
    "MlpPolicy", train_env,
    learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10,
    gamma=0.97, gae_lambda=0.95, clip_range=0.2,
    seed=RNG_SEED, verbose=1, device=device,
    tensorboard_log=str(MODELS / "tb"),
)
model.set_logger(logger)

# verbose=1 + progress_bar=True gives you both the SB3 tables and a live progress bar.
model.learn(total_timesteps=400_000, callback=eval_cb, progress_bar=True)

# Release caches so memory isn't held after training.
if device == "cuda":
    torch.cuda.empty_cache()
gc.collect()
print("Training complete. Best model saved to", MODELS)


## 7.0 Policy Evaluation & Optimal Trajectory

Roll the trained policy out across 200 episodes and collect the per-quarter trajectories, so we
can look at the *distribution* of outcomes rather than a single run. Five key plots follow:
margin trajectory (±IQR), inventory-led share trajectory, action heatmap, market share vs
do-nothing baseline, and breakeven quarter distribution.

In [ ]:

def rollout(model, n_episodes=200, deterministic=True):
    env = InstamartTransitionEnv()
    paths = []
    for ep in range(n_episodes):
        obs, _ = env.reset()
        done = False
        step_rows = []
        while not done:
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, term, trunc, info = env.step(int(action))
            step_rows.append({**info, "action": int(action), "reward": r, "episode": ep})
            done = term or trunc
        paths.append(pd.DataFrame(step_rows))
    return pd.concat(paths, ignore_index=True)

learned = rollout(model)
ppo_final = learned.sort_values("quarter").groupby("episode").tail(1)

print("PPO policy       :  median final margin = {:+.3f},  median reward = {:.1f}".format(
    ppo_final["margin"].median(), ppo_final["reward"].cumsum().median()))
print()
print("Improvement over always-Hold      : {:+.3f} margin".format(
    ppo_final["margin"].median() - always_hold["margin"].median()))
print("Improvement over always-Aggressive: {:+.3f} margin".format(
    ppo_final["margin"].median() - always_aggressive["margin"].median()))


In [ ]:

# --- Plot 1: median margin trajectory with IQR band ---
def band(df, col):
    g = df.groupby("quarter")[col]
    return g.median(), g.quantile(0.25), g.quantile(0.75)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

med, lo, hi = band(learned, "margin")
axes[0].plot(med.index, med.values * 100, color=SWIGGY, linewidth=2, marker="o", label="Median")
axes[0].fill_between(med.index, lo.values * 100, hi.values * 100, color=SWIGGY, alpha=0.18, label="IQR")
axes[0].axhline(0, color=NAVY, linestyle="--", linewidth=0.9)
axes[0].set_title("Learned policy: contribution-margin trajectory")
axes[0].set_xlabel("Quarter"); axes[0].set_ylabel("Contribution margin (%)")
axes[0].legend()

med, lo, hi = band(learned, "pct_inv")
axes[1].plot(med.index, med.values * 100, color=BLINKIT, linewidth=2, marker="o", label="Median")
axes[1].fill_between(med.index, lo.values * 100, hi.values * 100, color=BLINKIT, alpha=0.18, label="IQR")
axes[1].axhline(90, color=NAVY, linestyle="--", linewidth=0.9, label="Blinkit (90%)")
axes[1].set_title("Learned policy: how fast it pushes inventory-led share")
axes[1].set_xlabel("Quarter"); axes[1].set_ylabel("Inventory-led NOV (%)")
axes[1].legend()

plt.tight_layout()
plt.savefig(PROCESSED / "b6a_chart_learned_policy_trajectories.png", bbox_inches="tight")
plt.show()


In [ ]:

# --- Plot 2: action mix by quarter (what the policy actually does over time) ---
action_labels = {0: "Hold", 1: "Slow +5pp", 2: "Moderate +10pp", 3: "Aggressive +15pp", 4: "Retreat -5pp"}
mix = (learned.assign(action_name=learned["action"].map(action_labels))
       .groupby(["quarter", "action_name"]).size().unstack(fill_value=0))
mix = mix.div(mix.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(10, 4.5))
bottom = np.zeros(len(mix))
palette = {"Hold": "#AEB6C2", "Slow +5pp": "#85B7EB", "Moderate +10pp": "#378ADD",
           "Aggressive +15pp": BLINKIT, "Retreat -5pp": SWIGGY}
for col in ["Hold", "Slow +5pp", "Moderate +10pp", "Aggressive +15pp", "Retreat -5pp"]:
    if col in mix.columns:
        ax.bar(mix.index, mix[col], bottom=bottom, label=col, color=palette[col])
        bottom += mix[col].values
ax.set_title("What the learned policy does each quarter")
ax.set_xlabel("Quarter"); ax.set_ylabel("Share of episodes")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(PROCESSED / "b6a_chart_action_mix.png", bbox_inches="tight")
plt.show()


In [ ]:

# --- Plot 2b: action frequency heatmap (quarter × action → share of episodes) ---
action_labels = {0: "Hold", 1: "Slow +5pp", 2: "Mod +10pp", 3: "Aggr +15pp", 4: "Retreat"}
col_order     = ["Hold", "Slow +5pp", "Mod +10pp", "Aggr +15pp", "Retreat"]

heat = (learned.assign(action_name=learned["action"].map(action_labels))
        .groupby(["quarter", "action_name"]).size().unstack(fill_value=0))
heat = heat.div(heat.sum(axis=1), axis=0)
heat = heat.reindex(columns=[c for c in col_order if c in heat.columns])

fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(heat.T.values, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(heat.index)))
ax.set_xticklabels([f"Q{q+1}" for q in heat.index])
ax.set_yticks(range(len(heat.columns)))
ax.set_yticklabels(heat.columns)
plt.colorbar(im, ax=ax, label="Share of episodes")
for i in range(len(heat.columns)):
    for j in range(len(heat.index)):
        val = heat.T.values[i, j]
        ax.text(j, i, f"{val:.0%}", ha="center", va="center",
                fontsize=8, color="black" if val < 0.55 else "white")
ax.set_title("Action frequency heatmap — what the learned policy does each quarter")
ax.set_xlabel("Quarter")
plt.tight_layout()
plt.savefig(PROCESSED / "b6a_chart_action_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:

# --- Plot 3: quarters-to-breakeven distribution under the learned policy ---
def breakeven_quarter(group):
    pos = group[group["margin"] >= 0]
    return int(pos["quarter"].min()) if len(pos) else np.nan

bq = learned.groupby("episode").apply(breakeven_quarter, include_groups=False).dropna()
reached = len(bq) / learned["episode"].nunique()

fig, ax = plt.subplots(figsize=(7, 4))
if len(bq):
    ax.hist(bq, bins=range(0, 13), color=SWIGGY, edgecolor="white", align="left")
    ax.axvline(bq.median(), color=NAVY, linestyle="--", label=f"Median = Q{bq.median():.0f}")
    ax.legend()
ax.set_title(f"Quarters to breakeven under the learned policy\n"
             f"({reached:.0%} of episodes reach positive contribution margin within 3 years)")
ax.set_xlabel("Quarter of first positive contribution margin"); ax.set_ylabel("Episodes")
plt.tight_layout()
plt.savefig(PROCESSED / "b6a_chart_breakeven_distribution.png", bbox_inches="tight")
plt.show()

print(f"Share of episodes reaching breakeven within 12 quarters: {reached:.1%}")
if len(bq):
    print(f"Median quarter of breakeven (where achieved): Q{bq.median():.0f}")


In [ ]:

# --- Plot 4: market share trajectory — PPO vs Always-Hold baseline ---
# Re-collect per-step trajectories for the hold policy
# (run_fixed_policy in §5.1 only stores final states, not per-quarter paths)
_hold_steps = []
_env_h = InstamartTransitionEnv()
for _ep in range(200):
    _obs, _ = _env_h.reset()
    _done   = False
    while not _done:
        _obs, _r, _term, _trunc, _info = _env_h.step(0)  # always hold
        _hold_steps.append({**_info, "episode": _ep})
        _done = _term or _trunc
hold_traj = pd.DataFrame(_hold_steps)

fig, ax = plt.subplots(figsize=(10, 4.5))

# PPO policy
med_p, lo_p, hi_p = band(learned, "share")
ax.plot(med_p.index, med_p.values * 100, color=SWIGGY, linewidth=2,
        marker="o", label="PPO (learned policy)")
ax.fill_between(med_p.index, lo_p.values * 100, hi_p.values * 100,
                color=SWIGGY, alpha=0.18)

# Always-Hold baseline
med_h, lo_h, hi_h = band(hold_traj, "share")
ax.plot(med_h.index, med_h.values * 100, color="grey", linewidth=1.5,
        linestyle="--", marker="s", label="Always-Hold baseline")
ax.fill_between(med_h.index, lo_h.values * 100, hi_h.values * 100,
                color="grey", alpha=0.12)

ax.axhline(BASELINE["blinkit_market_share"] * 100, color=BLINKIT, linestyle=":",
           linewidth=1.2, label=f"Blinkit ({BASELINE['blinkit_market_share']:.0%})")
ax.axhline(BASELINE["market_share"] * 100, color=NAVY, linestyle="--",
           linewidth=0.8, alpha=0.7, label="Starting share (24%)")

ax.set_title("Market share trajectory: PPO learned policy vs do-nothing baseline")
ax.set_xlabel("Quarter")
ax.set_ylabel("Market share (%)")
ax.legend()
plt.tight_layout()
plt.savefig(PROCESSED / "b6a_chart_market_share_trajectory.png", bbox_inches="tight")
plt.show()

### 7.1 Plain-English Interpretation

Translate the charts above into the one or two sentences you would say to a hiring manager.
Because training has some stochasticity, describe the *pattern* rather than exact numbers:

- The learned policy almost never chooses **Retreat** and rarely sits on **Hold** for long — the
  model finds that the inventory-model margin uplift outweighs its capex cost, *given the assumed
  ₹80 cr-per-5pp transition cost*.
- It typically front-loads transition in the early-to-middle quarters, then eases off once
  inventory-led share approaches the Blinkit-like ceiling and lets the density→margin→capex
  reinforcing loop carry the rest.
- The improvement over the always-Hold baseline is the quantitative version of Branch 4’s
  qualitative finding: the inventory model is the single biggest structural lever.

**Crucial caveat:** this is the optimal pace *inside the model*. The single most load-bearing
assumption is `inv_model_total_bps` (240 bps), which has no individual public source. Section 8
stresses the reward weights; a real engagement would also stress this cost.

## 8.0 Ablation: Reward Weight Sensitivity

The reward weights (50/30/20 on margin/share/capex) are a modelling assumption. If the learned
policy flips completely when those weights move a little, that’s important to disclose.
Re-train quickly under three weightings and compare the resulting median final margin and the
average inventory-led share the policy settles on.

In [ ]:

weight_grid = [
    dict(w_margin=0.70, w_share=0.20, w_capex=0.10, label="Margin-focused"),
    dict(w_margin=0.50, w_share=0.30, w_capex=0.20, label="Base case"),
    dict(w_margin=0.30, w_share=0.50, w_capex=0.20, label="Share-focused"),
]

sens_rows = []
for cfg in weight_grid:
    e = InstamartTransitionEnv(config=cfg)
    # Use the same memory-friendly PPO settings as main training, but lighter budget for the sweep
    m = PPO(
        "MlpPolicy", e,
        learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10,
        gamma=0.97, seed=RNG_SEED, verbose=0, device=device
    )
    m.learn(total_timesteps=150_000)   # moderate budget for sensitivity sweep
    roll = rollout(m, n_episodes=100)
    fin = roll.sort_values("quarter").groupby("episode").tail(1)
    sens_rows.append(dict(
        scenario=cfg["label"],
        median_final_margin=round(fin["margin"].median(), 4),
        mean_final_inv_share=round(roll.groupby("episode").tail(1)["pct_inv"].mean(), 3),
        median_final_share=round(fin["share"].median(), 4),
    ))
    # Clear memory between runs to avoid hoarding RAM/GPU memory
    del m
    if device == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

sensitivity = pd.DataFrame(sens_rows)
sensitivity.to_csv(PROCESSED / "b6a_reward_weight_sensitivity.csv", index=False)
sensitivity


In [ ]:

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(sensitivity["scenario"], sensitivity["mean_final_inv_share"] * 100, color=[ZEPTO, SWIGGY, BLINKIT])
ax.set_ylabel("Mean final inventory-led share (%)")
ax.set_title("Does the optimal transition pace survive different reward weightings?")
for i, v in enumerate(sensitivity["mean_final_inv_share"]):
    ax.text(i, v * 100 + 1, f"{v*100:.0f}%", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig(PROCESSED / "b6a_chart_reward_sensitivity.png", bbox_inches="tight")
plt.show()


## 9.0 Results & Honest Limitations

**Verdict on Strategy 1:** within a simulation calibrated to disclosed aggregates, an RL agent
consistently prefers a fairly aggressive, front-loaded inventory-model transition over doing
nothing, and that preference is reasonably stable across reward weightings. This is the
*quantitative* echo of Branch 4’s qualitative conclusion that the inventory model is the biggest
structural lever — now with an explicit, defensible statement of *pace*.

**Limitations (the part that makes this credible rather than a toy):**

1. **`inv_model_total_bps` (240 bps) is an unsourced estimate.** It is the single most
   load-bearing assumption in the model; the whole policy could shift if the true value is very
   different. A real engagement would source this from Swiggy’s capex disclosures and stress-test
   it the way Section 8 stresses the reward weights.
2. **The density→margin relationship is calibrated to the n=1,143 *simulated* store network
   (Notebook 02), not real stores.** The functional form (log-linear above breakeven) is an
   assumption.
3. **The 75% shareholder-vote gate is treated as already resolved.** Modelling the governance
   approval itself would need a separate meta-layer; here we answer *“given approval, how fast?”*
4. **Real inventory transitions involve supplier contracts, warehouse build-out, and cold-chain
   logistics** — none of which are in this model. The environment abstracts all of that into a
   single capex cost.

The one-line framing for a recruiter: *“I used reinforcement learning to find the optimal pace for
the inventory-model transition inside a calibrated simulation, then stress-tested the result
against its own reward assumptions and was explicit about the one cost estimate that drives
everything.”*